In [4]:
import pandas as pd
import sqlite3
import numpy as np

RAW_DATA_PATH = r"C:\Users\reddy\Downloads\archive (1)\synthetic_telemetry_data.csv"
DB_NAME = "cat_fleet_medallion.db"

# ── Bronze Layer ──────────────────────────────────────────────────────────────
with sqlite3.connect(DB_NAME) as conn:
    df_raw = pd.read_csv(RAW_DATA_PATH)
    df_raw = df_raw.rename(columns={'vehicle_id': 'machine_id'})
    df_raw.to_sql("bronze_telematics", conn, if_exists="replace", index=False)
    print(f"Bronze loaded: {len(df_raw)} rows")

    # ── Silver Layer ──────────────────────────────────────────────────────────
    df_silver = pd.read_sql("SELECT * FROM bronze_telematics", conn)

    # Parse timestamps
    df_silver['timestamp'] = pd.to_datetime(df_silver['timestamp'])
    df_silver['failure_date'] = pd.to_datetime(df_silver['failure_date'])

    # Map granular failure types to categories
    failure_mapping = {
        'Engine Overheat':    'Engine',
        'Low Oil Pressure':   'Engine',
        'Excessive Vibration':'Engine',
        'Brake Pad Worn':     'Brake',
        'Low Brake Fluid':    'Brake',
        'Brake Overheat':     'Brake',
        'Battery Dead':       'Battery',
        'Battery Drain':      'Battery',
        'Low Battery Voltage':'Battery',
        'No Failure':         'Healthy'
    }

    # Log any unmapped values before filling
    unmapped = df_silver[~df_silver['failure_type'].isin(failure_mapping)]['failure_type'].unique()
    if len(unmapped) > 0:
        print(f"Unmapped failure types (defaulting to Healthy): {unmapped}")

    df_silver['clean_failure_type'] = df_silver['failure_type'].map(failure_mapping).fillna('Healthy')

    # Idling flag
    df_silver['is_idling'] = np.where(
        (df_silver['vehicle_speed_kph'] == 0) & (df_silver['engine_rpm'] > 0), 1, 0
    )

    # First failure timestamp per machine (fixed)
    first_failure = (
        df_silver[df_silver['clean_failure_type'] != 'Healthy']
        .groupby('machine_id')['timestamp']
        .min()
        .rename('failure_timestamp')
    )
    df_silver = df_silver.merge(first_failure, on='machine_id', how='left')

    # Hours until that first failure
    df_silver['hours_until_failure'] = (
        df_silver['failure_timestamp'] - df_silver['timestamp']
    ).dt.total_seconds() / 3600

    # Target label: flag rows within 12h lookback window before failure
    LOOKBACK_WINDOW = 12
    df_silver['final_target_label'] = 'Healthy'

    at_risk_mask = (
        (df_silver['hours_until_failure'] >= 0) &
        (df_silver['hours_until_failure'] <= LOOKBACK_WINDOW) &
        (df_silver['clean_failure_type'] != 'Healthy')
    )
    df_silver.loc[at_risk_mask, 'final_target_label'] = df_silver.loc[at_risk_mask, 'clean_failure_type']

    df_silver.to_sql("silver_telematics", conn, if_exists="replace", index=False)
    print(f"Silver loaded: {len(df_silver)} rows")

    # ── Gold Layer ────────────────────────────────────────────────────────────
    df_gold = pd.read_sql("SELECT * FROM silver_telematics", conn)

    # Sort for rolling features
    df_gold = df_gold.sort_values(['machine_id', 'timestamp']).reset_index(drop=True)

    # Rolling averages (3-reading window per machine)
    roll_cols = ['engine_temp_celsius', 'engine_rpm', 'oil_pressure_psi',
                 'battery_voltage_v', 'vibration_ms2']
    
    existing_roll_cols = [c for c in roll_cols if c in df_gold.columns]
    for col in existing_roll_cols:
        df_gold[f'{col}_roll3_mean'] = (
            df_gold.groupby('machine_id')[col]
            .transform(lambda x: x.rolling(3, min_periods=1).mean())
        )

    # Encode target label for ML
    label_map = {'Healthy': 0, 'Engine': 1, 'Brake': 2, 'Battery': 3}
    df_gold['target_encoded'] = df_gold['final_target_label'].map(label_map).fillna(0).astype(int)

    df_gold.to_sql("gold_telematics", conn, if_exists="replace", index=False)
    print(f"Gold loaded: {len(df_gold)} rows")

    # ── Verification Audit ────────────────────────────────────────────────────
    print("\n── Silver label distribution ──")
    audit_silver = pd.read_sql(
        "SELECT final_target_label, COUNT(*) as row_count FROM silver_telematics GROUP BY final_target_label",
        conn
    )
    print(audit_silver.to_string(index=False))

    print("\n── Gold encoded label distribution ──")
    audit_gold = pd.read_sql(
        "SELECT target_encoded, final_target_label, COUNT(*) as row_count FROM gold_telematics GROUP BY target_encoded, final_target_label",
        conn
    )
    print(audit_gold.to_string(index=False))

print("\n All three layers written to", DB_NAME)

Bronze loaded: 1970 rows
Silver loaded: 1970 rows
Gold loaded: 1970 rows

── Silver label distribution ──
final_target_label  row_count
           Battery         10
             Brake         11
            Engine          4
           Healthy       1945

── Gold encoded label distribution ──
 target_encoded final_target_label  row_count
              0            Healthy       1945
              1             Engine          4
              2              Brake         11
              3            Battery         10

 All three layers written to cat_fleet_medallion.db
